In [1]:
!pip install pyspark

from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
import os

# Initialize Spark Session
spark = SparkSession.builder.appName("Week6_Spark_Assignment").getOrCreate()

# Create directories for our dummy data to satisfy file path questions
os.makedirs("data", exist_ok=True)
os.makedirs("path/to", exist_ok=True)

# Define data matching the assignment constraints
data = [
    (1, "Electronics", "1200", "OldProductA", "Completed", 1200, 1000, "101", "North", "High"),
    (2, "Clothing", "45", "OldProductB", "Pending", 45, 40, "102", "South", "Low"),
    (3, "Electronics", "850", "OldProductC", "Completed", 1500, 700, None, "East", "High"),
    (4, "Home", "1500", "OldProductD", "Completed", 1500, 1300, "104", "North", "Low")
]

schema = StructType([
    StructField("product_id", IntegerType(), True),
    StructField("category", StringType(), True),
    StructField("price", StringType(), True),
    StructField("old_name", StringType(), True),
    StructField("status", StringType(), True),
    StructField("amount", IntegerType(), True),
    StructField("base_price", IntegerType(), True),
    StructField("user_id", StringType(), True),
    StructField("region", StringType(), True),
    StructField("priority", StringType(), True)
])

df = spark.createDataFrame(data, schema=schema)
df_orders = df

df.write.mode("overwrite").option("header", "true").csv("data/source.csv")
df.write.mode("overwrite").parquet("path/to/input")

print("Environment Setup Complete. Dummy data generated.")
df.show()

Environment Setup Complete. Dummy data generated.
+----------+-----------+-----+-----------+---------+------+----------+-------+------+--------+
|product_id|   category|price|   old_name|   status|amount|base_price|user_id|region|priority|
+----------+-----------+-----+-----------+---------+------+----------+-------+------+--------+
|         1|Electronics| 1200|OldProductA|Completed|  1200|      1000|    101| North|    High|
|         2|   Clothing|   45|OldProductB|  Pending|    45|        40|    102| South|     Low|
|         3|Electronics|  850|OldProductC|Completed|  1500|       700|   NULL|  East|    High|
|         4|       Home| 1500|OldProductD|Completed|  1500|      1300|    104| North|     Low|
+----------+-----------+-----+-----------+---------+------+----------+-------+------+--------+



### Q1: Explain the roles of the Driver, Cluster Manager, and Executor in a Spark application.

* **Driver:** The central coordinator of the Spark application. It runs the `main()` function, converts the user's code into tasks, creates the DAG (Directed Acyclic Graph), and negotiates with the Cluster Manager to allocate resources.
* **Cluster Manager:** The resource manager (like YARN, Mesos, or Spark Standalone) that allocates cluster resources (CPU, memory) to the Spark application as requested by the Driver.
* **Executor:** The worker nodes that actually execute the tasks assigned by the Driver. They run the data processing code, store data in memory (caching), and return the results back to the Driver.

### Q2: How does Spark's Lazy Evaluation strategy improve performance when processing large datasets?

Lazy evaluation means Spark does not immediately execute transformations when they are called. Instead, it builds a Lineage Graph (DAG) of all the requested transformations. Execution only begins when an Action (like `.show()` or `.collect()`) is called.
This improves performance because the Catalyst Optimizer can look at the entire chain of transformations before running anything. It can optimize the execution plan, skip unnecessary steps, and combine operations to minimize disk I/O and data shuffling.

In [2]:
# Q3: Write a Spark command to read a CSV file located at "data/source.csv", ensuring the first row is treated as a header and inferSchema is enabled.

df_csv = spark.read.csv("data/source.csv", header=True, inferSchema=True)
df_csv.show(2)

+----------+-----------+-----+-----------+---------+------+----------+-------+------+--------+
|product_id|   category|price|   old_name|   status|amount|base_price|user_id|region|priority|
+----------+-----------+-----+-----------+---------+------+----------+-------+------+--------+
|         1|Electronics| 1200|OldProductA|Completed|  1200|      1000|    101| North|    High|
|         2|   Clothing|   45|OldProductB|  Pending|    45|        40|    102| South|     Low|
+----------+-----------+-----+-----------+---------+------+----------+-------+------+--------+
only showing top 2 rows


### Q4: What is the difference between CSV and Parquet in terms of storage (row-based vs. columnar) and why does it matter for performance?

* **CSV (Row-Based):** Stores data row by row. It is human-readable but inefficient for querying specific columns because the engine must scan entire rows to extract the desired column data. It does not support schema enforcement inherently.
* **Parquet (Columnar):** Stores data column by column. It is highly compressed and optimized for analytical queries.
* **Performance Impact:** Parquet significantly improves performance because of **column pruning** (Spark only reads the specific columns requested in a query, skipping the rest) and better compression, drastically reducing disk I/O and memory usage compared to CSV.

In [3]:
# Q5: Given a DataFrame df, write a query to select the columns product_id and price where the category is 'Electronics'.

df.filter(col("category") == "Electronics") \
  .select("product_id", "price") \
  .show()

+----------+-----+
|product_id|price|
+----------+-----+
|         1| 1200|
|         3|  850|
+----------+-----+



In [4]:
# Q6: Write the code to "revise" a DataFrame by renaming the column old_name to new_name and casting the price column from a String to a Double.

df_revised = df.withColumnRenamed("old_name", "new_name") \
               .withColumn("price", col("price").cast("double"))

df_revised.printSchema()
df_revised.select("new_name", "price").show(2)

root
 |-- product_id: integer (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- new_name: string (nullable = true)
 |-- status: string (nullable = true)
 |-- amount: integer (nullable = true)
 |-- base_price: integer (nullable = true)
 |-- user_id: string (nullable = true)
 |-- region: string (nullable = true)
 |-- priority: string (nullable = true)

+-----------+------+
|   new_name| price|
+-----------+------+
|OldProductA|1200.0|
|OldProductB|  45.0|
+-----------+------+
only showing top 2 rows


In [5]:
# Q6: Write the code to "revise" a DataFrame by renaming the column old_name to new_name and casting the price column from a String to a Double.

df_revised = df.withColumnRenamed("old_name", "new_name") \
               .withColumn("price", col("price").cast("double"))

df_revised.printSchema()
df_revised.select("new_name", "price").show(2)

root
 |-- product_id: integer (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- new_name: string (nullable = true)
 |-- status: string (nullable = true)
 |-- amount: integer (nullable = true)
 |-- base_price: integer (nullable = true)
 |-- user_id: string (nullable = true)
 |-- region: string (nullable = true)
 |-- priority: string (nullable = true)

+-----------+------+
|   new_name| price|
+-----------+------+
|OldProductA|1200.0|
|OldProductB|  45.0|
+-----------+------+
only showing top 2 rows


### Q7: How does Spark use the Lineage Graph (DAG) to provide fault tolerance if a worker node fails?

Spark tracks all transformations applied to a base dataset using a Lineage Graph (Directed Acyclic Graph). If a worker node (Executor) fails and a partition of data is lost, Spark does not need to replicate data across nodes like Hadoop. Instead, it looks at the DAG and simply recomputes that specific lost partition from the original source data using the recorded sequence of transformations.

In [6]:
# Q8: Write a query to filter a DataFrame df_orders for rows where the status is 'Completed' AND the amount is greater than 1000.

df_orders.filter((col("status") == "Completed") & (col("amount") > 1000)).show()

+----------+-----------+-----+-----------+---------+------+----------+-------+------+--------+
|product_id|   category|price|   old_name|   status|amount|base_price|user_id|region|priority|
+----------+-----------+-----+-----------+---------+------+----------+-------+------+--------+
|         1|Electronics| 1200|OldProductA|Completed|  1200|      1000|    101| North|    High|
|         3|Electronics|  850|OldProductC|Completed|  1500|       700|   NULL|  East|    High|
|         4|       Home| 1500|OldProductD|Completed|  1500|      1300|    104| North|     Low|
+----------+-----------+-----+-----------+---------+------+----------+-------+------+--------+



### Q9: Explain the concept of Predicate Pushdown in Parquet and how it affects the amount of data loaded into memory.
Predicate Pushdown (or filter pushdown) is an optimization technique where filtering conditions (predicates) are pushed down to the storage layer (like a Parquet file) rather than loading all the data into memory and filtering it in Spark.
Because Parquet stores metadata (like min/max values for each chunk), Spark can check this metadata and completely skip reading chunks of data that do not satisfy the filter condition. This drastically reduces the amount of data read from disk and loaded into memory, speeding up the query.

In [7]:
# Q10: Write a code snippet to add a new column final_price which is the base_price multiplied by 1.18 (18% tax).

df_taxed = df.withColumn("final_price", col("base_price") * 1.18)
df_taxed.select("base_price", "final_price").show()

+----------+------------------+
|base_price|       final_price|
+----------+------------------+
|      1000|            1180.0|
|        40|47.199999999999996|
|       700|             826.0|
|      1300|            1534.0|
+----------+------------------+



### Q11: What is the difference between Transformations and Actions?Provide two examples of each.
* **Transformations:** Operations that create a new DataFrame from an existing one. They are *lazy*, meaning they don't execute immediately but just add to the DAG.
  * *Examples:* `.select()`, `.filter()`, `.groupBy()`
* **Actions:** Operations that trigger the actual execution of the DAG to return a result to the Driver or write data to storage.
  * *Examples:* `.show()`, `.collect()`, `.count()`, `.write()`

In [8]:
# Q12: Write the Spark command to load a Parquet file from "path/to/input", filter out any rows where user_id is null, and save the result as a CSV at "path/to/output".

# Read Parquet
df_parquet = spark.read.parquet("path/to/input")

# Filter nulls
df_filtered = df_parquet.filter(col("user_id").isNotNull())

# Write to CSV
df_filtered.write.mode("overwrite").option("header", "true").csv("path/to/output")

print("Data successfully filtered and written to path/to/output")

Data successfully filtered and written to path/to/output


### Q13: In Spark Architecture, what is the difference between Client Mode and Cluster Mode?

* **Client Mode:** The Driver process runs on the client machine that submitted the application (e.g., your laptop or an edge node). If the client disconnects or shuts down, the Spark job fails. Useful for interactive development and debugging.
* **Cluster Mode:** The Driver process runs on one of the worker nodes within the cluster, managed by the Cluster Manager. Once the job is submitted, the client can disconnect without affecting the running application. Used for production environments.

In [9]:
# Q14: Write a query to filter a dataset for rows where the region is 'North' OR the priority is 'High'.

df.filter((col("region") == "North") | (col("priority") == "High")).show()

+----------+-----------+-----+-----------+---------+------+----------+-------+------+--------+
|product_id|   category|price|   old_name|   status|amount|base_price|user_id|region|priority|
+----------+-----------+-----+-----------+---------+------+----------+-------+------+--------+
|         1|Electronics| 1200|OldProductA|Completed|  1200|      1000|    101| North|    High|
|         3|Electronics|  850|OldProductC|Completed|  1500|       700|   NULL|  East|    High|
|         4|       Home| 1500|OldProductD|Completed|  1500|      1300|    104| North|     Low|
+----------+-----------+-----+-----------+---------+------+----------+-------+------+--------+



### Q15: When exploring a dataset, why is it safer to use .show(5) instead of .collect() on a multi-terabyte dataset?

* `.show(5)` only computes and returns 5 rows to the console.
* `.collect()` brings the *entire dataset* from all Executor nodes back to the single Driver node's memory.
If you run `.collect()` on a multi-terabyte dataset, it will easily exceed the Driver's available memory, causing an OutOfMemory (OOM) error and crashing the Spark application.

## Brief Insights: Spark Performance & Architecture

Building this data pipeline highlights several critical architectural and performance features of Apache Spark:

**1. Architectural Resilience (The DAG):**
Spark’s architecture separates the cluster management from the actual execution. Because the **Driver** maintains a Directed Acyclic Graph (DAG) of all operations, the system is inherently fault-tolerant. If an **Executor** fails while processing our filtering tasks, Spark doesn't lose the data; it simply looks at the Lineage Graph and recomputes the lost partition from the source `source.csv` or Parquet file.

**2. The Power of Lazy Evaluation:**
By strictly separating **Transformations** (like `.filter()` and `.withColumn()`) from **Actions** (like `.show()` and `.write()`), Spark saves immense computational power. When we chain operations—such as filtering for 'Electronics', renaming a column, and casting a data type—Spark's Catalyst Optimizer reviews the entire DAG before executing anything, ensuring data is moved (shuffled) across the network as little as possible.

**3. Storage Drives Performance (Parquet vs. CSV):**
The assignment demonstrates reading from both CSV and Parquet. For large-scale data engineering, Parquet is vastly superior. Because it is a **columnar format**, operations that only require specific columns (like selecting `product_id` and `price`) benefit from **column pruning**—Spark entirely skips reading the irrelevant columns from the disk. 

**4. Predicate Pushdown:**
Combined with Parquet, filtering operations (like `status = 'Completed'`) utilize **Predicate Pushdown**. Spark pushes this filter down to the storage layer, using Parquet's metadata to skip entire blocks of data that don't match the condition, drastically reducing disk I/O and preventing unnecessary data from flooding the Executor's memory.